In [2]:
import numpy as np  # type: ignore
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import Ridge

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


ruta = r'C:\Users\jthow\iCloudDrive\Documents\3_Maestria_Estadistica_UNINORTE\3_Tercer_Semestre\Machine_Learning\tornados.csv.zip'  
df = pd.read_csv(ruta)  
df['loss'] = df['loss'].replace(0, pd.NA)
df['loss'] = df['loss'].interpolate(method='linear')
df['mag'] = df['mag'].fillna(df['mag'].mean())
df.isnull().sum()

om              0
yr              0
mo              0
dy              0
date            0
time            0
tz              0
datetime_utc    0
st              0
stf             0
mag             0
inj             0
fat             0
loss            0
slat            0
slon            0
elat            0
elon            0
len             0
wid             0
ns              0
sn              0
f1              0
f2              0
f3              0
f4              0
fc              0
dtype: int64

In [ ]:
pip install mglearn

In [3]:
# Crear la columna 'mortality' en el DataFrame original
df['mortality'] = df['fat'].apply(lambda x: 0 if x == 0 else 1)

# Renombrar el DataFrame a 'mortality_target'
mortality_target = df
import numpy as np

# Crear la columna 'mortality' con 0 si 'fat' es 0, y 1 si 'fat' es mayor que 0
df['mortality'] = np.where(df['fat'] == 0, 0, 1)

# Contar la cantidad de ceros y unos
print("Cantidad de ceros:", (df['mortality'] == 0).sum())
print("Cantidad de unos:", (df['mortality'] == 1).sum())

# Asignar el DataFrame modificado a 'tornados.target'
tornados_target = df

Cantidad de ceros: 67120
Cantidad de unos: 1573


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier  # Importar HistGradientBoostingClassifier

# Definir las variables X y y
X = tornados_target[['om', 'yr', 'mo', 'dy', 'stf', 'mag', 'inj', 'loss', 'slat', 'slon', 'elat', 'elon', 'len', 'wid', 'ns', 'sn', 'f1', 'f2', 'f3', 'f4']]
y = df['mortality']

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Inicializar el modelo HistGradientBoostingClassifier
hist_gb = HistGradientBoostingClassifier(max_iter=100, random_state=42)

# Entrenar el modelo
hist_gb.fit(X_train, y_train)

# Evaluación del modelo
train_score = hist_gb.score(X_train, y_train)
test_score = hist_gb.score(X_test, y_test)

print(f"Training score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")


Training score: 0.99
Test score: 0.98


## Metricas Para Hist Gradient Boosting Classifier

In [6]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier  # Cambiar por HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    accuracy_score, 
    f1_score, 
    roc_auc_score
)
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera
from joblib import dump
from time import time  # Importar time para medir el tiempo de entrenamiento

# ------------------------
# Paso 2: Cargar los datos
# ------------------------
# Suponiendo que X_train, X_test, y_train, y_test ya están definidas

# ------------------------
# Paso 3: Definir el pipeline y el grid de hiperparámetros
# ------------------------
# Definir el pipeline para el modelo de HistGradientBoostingClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

# Definir el grid de hiperparámetros para HistGradientBoostingClassifier
param_grid = {
    'model__max_iter': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__learning_rate': [0.05, 0.1, 0.2]
}

# ------------------------
# Paso 4: Entrenar el modelo con GridSearchCV
# ------------------------
# Medir el tiempo de entrenamiento
start_time = time()

# Realizar GridSearchCV para el modelo
grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Calcular el tiempo total de entrenamiento
training_time = time() - start_time

# Obtener el mejor modelo
best_model = grid_search.best_estimator_

# Guardar el modelo entrenado
dump(grid_search, 'grid_hgbc.joblib')

# ------------------------
# Paso 5: Hacer predicciones con el mejor modelo
# ------------------------
y_pred = best_model.predict(X_test)

# ------------------------
# Paso 6: Calcular las métricas de clasificación
# ------------------------
# Convertir las predicciones y valores verdaderos a variables de clasificación binaria
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred)


# ------------------------
# Paso 7: Crear DataFrame con los resultados
# ------------------------
resultados_hgbc = pd.DataFrame({
    'Modelo': ['HistGradientBoostingClassifier optimizado (GridSearchCV)'],
    'Precision': [f"{precision:.2f}"],
    'Recall': [f"{recall:.2f}"],
    'Accuracy': [f"{accuracy:.2f}"],
    'F1-Score': [f"{f1:.2f}"],
    'AUC': [f"{auc:.2f}"],
    'CPU time (s)': [round(training_time, 2)]  # Incluir el tiempo de entrenamiento
})

# ------------------------
# Paso 8: Mostrar los resultados
# ------------------------
print("Métricas para el modelo de HistGradientBoostingClassifier:")
display(resultados_hgbc)


Métricas para el modelo de HistGradientBoostingClassifier:


,Modelo,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,HistGradientBoostingClassifier optimizado (Gri...,0.72,0.37,0.98,0.49,0.68,33.63


## Tabla de Comparación de metricas de clasificación

| Modelo                                                       | Precision | Recall | Accuracy | F1-Score | AUC   | CPU time (s) |
|--------------------------------------------------------------|-----------|--------|----------|----------|-------|--------------|
| Naive Bayes + GridSearch                                      | 0.33      | 0.67   | 0.94     | 0.44     | 0.91  | 0.61         |
| KNN optimizado (GridSearchCV)                                | 0.97      | 0.98   | 0.98     | 0.98     | 0.97  | 55.73        |
| Logistic L1                                                  | 0.89      | 0.90   | 0.90     | 0.89     | 0.87  | N/A          |
| Logistic L2                                                  | 0.86      | 0.89   | 0.89     | 0.86     | 0.80  | N/A          |
| RandomForestRegressor optimizado (GridSearchCV)              | 0.02      | 1.00   | 0.02     | 0.04     | 0.50  | 1406.24      |
| XGBRegressor                                                 | 0.03      | 0.99   | 0.21     | 0.05     | 0.59  | N/A          |
| SVM (kernel linear, sin SMOTE)                               | 1.00      | 0.08   | 0.97     | 0.15     | 0.90  | 38.08        |
| SVM (kernel linear, class_weight=balanced)                   | 0.23      | 0.84   | 0.90     | 0.37     | 0.94  | 196.96       |
| HistGradientBoostingClassifier optimizado (GridSearchCV)     | 0.72      | 0.37   | 0.98     | 0.49     | 0.68  | 33.63        |


In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_fscore_support, 
    accuracy_score, 
    roc_auc_score
)
from joblib import dump
from time import time

def train_hist_gradient_boosting_classifier(X_train, X_test, y_train, y_test):
    """
    Entrenar y evaluar un HistGradientBoostingClassifier con GridSearchCV.
    
    Parámetros:
    X_train, X_test: Features de entrenamiento y prueba
    y_train, y_test: Etiquetas de entrenamiento y prueba
    
    Retorna:
    DataFrame con métricas de rendimiento del modelo
    """
    # Definir pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', HistGradientBoostingClassifier(random_state=42))
    ])
    
    # Grid de hiperparámetros más exhaustivo y eficiente
    param_grid = {
        'model__max_iter': [100, 200, 300],
        'model__max_depth': [None, 10, 20],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__min_samples_leaf': [20, 30],
        'model__l2_regularization': [0, 0.1]
    }
    
    # Configuración de cross-validation estratificada
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Medir tiempo de entrenamiento
    start_time = time()
    
    # Realizar GridSearchCV
    grid_search = GridSearchCV(
        pipeline, 
        param_grid, 
        cv=cv, 
        n_jobs=-1, 
        scoring='balanced_accuracy',
        verbose=0
    )
    grid_search.fit(X_train, y_train)
    
    # Calcular tiempo de entrenamiento
    training_time = time() - start_time
    
    # Obtener mejor modelo
    best_model = grid_search.best_estimator_
    
    # Guardar modelo
    dump(grid_search, 'grid_hgbc_optimized.joblib')
    
    # Predicciones
    y_pred = best_model.predict(X_test)
    
    # Calcular métricas
    try:
        # Calcular precisión, recall, y F1-score
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average='weighted'
        )
        
        # Calcular AUC de manera robusta
        try:
            # Intentar calcular probabilidades para AUC
            y_pred_proba = best_model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
        except Exception:
            # Si falla, establecer AUC como None
            auc = None
        
        # Crear DataFrame de resultados
        resultados_hgbc = pd.DataFrame({
            'Modelo': ['HistGradientBoostingClassifier optimizado'],
            'Precision': [f"{precision:.2f}"],
            'Recall': [f"{recall:.2f}"],
            'Accuracy': [f"{accuracy_score(y_test, y_pred):.2f}"],
            'F1-Score': [f"{f1:.2f}"],
            'AUC': [f"{auc:.2f}" if auc is not None else "N/A"],
            'CPU time (s)': [round(training_time, 2)]
        })
        
        return resultados_hgbc
    
    except Exception as e:
        print(f"Error al calcular métricas: {e}")
        return None

# Uso del método (asumiendo que X_train, X_test, y_train, y_test están definidos)
resultados_hgbc = train_hist_gradient_boosting_classifier(X_train, X_test, y_train, y_test)

# Mostrar resultados
print("Métricas para el modelo de HistGradientBoostingClassifier:")
display(resultados_hgbc)

Métricas para el modelo de HistGradientBoostingClassifier:


,Modelo,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,HistGradientBoostingClassifier optimizado,0.98,0.98,0.98,0.98,N/A,258.1


| Modelo                                                   | Precision | Recall | Accuracy | F1-Score | AUC  | CPU time (s) |
|----------------------------------------------------------|-----------|--------|----------|----------|------|--------------|
| Naive Bayes + GridSearch                                  | 0.33      | 0.67   | 0.94     | 0.44     | 0.91 | 0.61         |
| KNN optimizado (GridSearchCV)                             | 0.97      | 0.98   | 0.98     | 0.98     | 0.97 | 55.73        |
| Logistic L1                                              | 0.89      | 0.90   | 0.90     | 0.89     | 0.87 | N/A          |
| Logistic L2                                              | 0.86      | 0.89   | 0.89     | 0.86     | 0.80 | N/A          |
| RandomForestRegressor optimizado (GridSearchCV)          | 0.02      | 1.00   | 0.02     | 0.04     | 0.50 | 1406.24      |
| XGBRegressor                                             | 0.03      | 0.99   | 0.21     | 0.05     | 0.59 | N/A          |
| SVM (kernel linear, sin SMOTE)                           | 1.00      | 0.08   | 0.97     | 0.15     | 0.90 | 38.08        |
| SVM (kernel linear, class_weight=balanced)               | 0.23      | 0.84   | 0.90     | 0.37     | 0.94 | 196.96       |
| HistGradientBoostingClassifier optimizado (GridSearchCV) | 0.72      | 0.37   | 0.98     | 0.49     | 0.68 | 33.63        |
| **HistGradientBoostingClassifier optimizado**            | 0.98      | 0.98   | 0.98     | 0.98     | 0.96 | 371.55       |


In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_fscore_support, 
    accuracy_score, 
    roc_auc_score
)
from joblib import dump
from time import time

def train_hist_gradient_boosting_classifier(X_train, X_test, y_train, y_test):
    """
    Entrenar y evaluar un HistGradientBoostingClassifier con GridSearchCV.
    
    Parámetros:
    X_train, X_test: Features de entrenamiento y prueba
    y_train, y_test: Etiquetas de entrenamiento y prueba
    
    Retorna:
    DataFrame con métricas de rendimiento del modelo
    """
    # Definir pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', HistGradientBoostingClassifier(random_state=42))
    ])
    
    # Grid de hiperparámetros más exhaustivo y eficiente
    param_grid = {
        'model__max_iter': [100, 200, 300],
        'model__max_depth': [None, 10, 20],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__min_samples_leaf': [20, 30],
        'model__l2_regularization': [0, 0.1]
    }
    
    # Configuración de cross-validation estratificada
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Medir tiempo de entrenamiento
    start_time = time()
    
    # Realizar GridSearchCV
    grid_search = GridSearchCV(
        pipeline, 
        param_grid, 
        cv=cv, 
        n_jobs=-1, 
        scoring='balanced_accuracy',
        verbose=0
    )
    grid_search.fit(X_train, y_train)
    
    # Calcular tiempo de entrenamiento
    training_time = time() - start_time
    
    # Obtener mejor modelo
    best_model = grid_search.best_estimator_
    
    # Guardar modelo
    dump(grid_search, 'grid_hgbc_optimized.joblib')
    
    # Predicciones
    y_pred = best_model.predict(X_test)
    
    # Calcular métricas
    try:
        # Calcular precisión, recall, y F1-score
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average='weighted'
        )
        
        # Verificar el número de clases para AUC
        unique_classes = np.unique(y_test)
        
        # Calcular AUC de manera robusta
        if len(unique_classes) == 2:
            # Para problemas binarios
            y_pred_proba = best_model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_pred_proba)
        elif len(unique_classes) > 2:
            # Para problemas multiclase
            y_pred_proba = best_model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
        else:
            auc = None
        
        # Crear DataFrame de resultados
        resultados_hgbc = pd.DataFrame({
            'Modelo': ['HistGradientBoostingClassifier optimizado'],
            'Precision': [f"{precision:.2f}"],
            'Recall': [f"{recall:.2f}"],
            'Accuracy': [f"{accuracy_score(y_test, y_pred):.2f}"],
            'F1-Score': [f"{f1:.2f}"],
            'AUC': [f"{auc:.2f}" if auc is not None else "N/A"],
            'CPU time (s)': [round(training_time, 2)]
        })
        
        return resultados_hgbc
    
    except Exception as e:
        print(f"Error al calcular métricas: {e}")
        return None

# Uso del método (asumiendo que X_train, X_test, y_train, y_test están definidos)
resultados_hgbc = train_hist_gradient_boosting_classifier(X_train, X_test, y_train, y_test)

# Mostrar resultados
print("Métricas para el modelo de HistGradientBoostingClassifier:")
display(resultados_hgbc)

Métricas para el modelo de HistGradientBoostingClassifier:


,Modelo,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,HistGradientBoostingClassifier optimizado,0.98,0.98,0.98,0.98,0.96,371.55
